# Walking-Based User Identification (Pocket Accelerometer / Gyroscope) — GPU Pipeline

**Task:** given a short window of inertial-sensor data recorded while a person walks (phone in pocket), predict **which user** it is (75 classes).

**What the data looks like (checked on your files)**

| Item | Value |
|---|---|
| `combined_main_ACC.csv` | 2.10 M rows, accelerometer only, ~200 Hz |
| `combined_main_RAW.csv` (inside the zip) | 2.10 M `acc` + 2.09 M `gyro` + 1.05 M `magn` rows (long format: `sensor_type, data_x/y/z`) |
| Users / recordings | 75 users, 126 recordings (70 users have **one** recording, 5 users have several) |
| Conditions | `hybrid_tests` and `mobile_in_hand`; sub-conditions e.g. `relaxed_arm/corridor`, `stairs` |
| Timestamps | `local_timestamp` is in **nanoseconds**, slightly out of order inside files → must sort |

**Pipeline**

1. Load → sort by timestamp → drop duplicates → split at time gaps
2. Resample to a uniform grid (100 Hz) → low-pass filter (20 Hz) → add magnitude channels
3. Sliding windows (2 s, 0.5 s stride)
4. **Leak-free split**: inside each recording the *first 70 %* in time → train, *next 15 %* → val, *last 15 %* → test, with gaps of whole windows removed between blocks so no sample overlaps (a random window shuffle would leak overlapping samples and give fake ~100 % accuracy)
5. GPU hand-crafted features (time, frequency, gait-periodicity) → classical ML on GPU (XGBoost-GPU, cuML RF / SVM / kNN / LogReg, torch MLP)
6. Deep learning on GPU (1D-ResNet, CNN-BiGRU) with AMP + augmentation
7. Soft-voting ensemble, block-voting, per-user accuracy, confusion matrix, **cross-recording sanity check**, saved artefacts

> **Important honesty note (read before trusting the accuracy).** 70/75 users have only one recording, so a test window comes from the *same recording* as the training windows. The model can therefore also exploit recording-specific cues (exact pocket orientation, sensor bias) and not only the person's gait. Section 9 runs a harder *cross-recording* test on the users who have multiple recordings so you can see the gap.

## 0. Setup (Colab / any CUDA machine)

In [ ]:
# Uncomment on a fresh Colab GPU runtime.
# xgboost is usually preinstalled. cuML (GPU RF/SVM/kNN/LogReg) is optional:
# !pip -q install xgboost scikit-learn scipy matplotlib
# !pip -q install --extra-index-url=https://pypi.nvidia.com "cuml-cu12==25.*"   # optional, needs restart

In [ ]:
import os, gc, time, zipfile, warnings, json, math, copy
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.signal import butter, sosfiltfilt
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---------------- GPU REQUIREMENT ----------------
REQUIRE_GPU = True            # keep True for real runs. (False only to dry-run on CPU)
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No CUDA GPU found. Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

## 1. Configuration

In [ ]:
# ------------------------------ PATHS ------------------------------
import pathlib

def _find_csv(filename):
    """Locate a CSV by trying several candidate directories."""
    candidates = [
        pathlib.Path(filename),                        # relative to CWD
        pathlib.Path(r'd:\Gait Dataset\Code\walking-recognition-dataset-77people') / filename,
        pathlib.Path.cwd() / filename,
        pathlib.Path.cwd() / "walking-recognition-dataset-77people" / filename,
    ]
    for p in candidates:
        if p.is_file():
            return str(p.resolve())
    raise FileNotFoundError(f"Cannot locate {filename}. Add its folder to candidates above.")

ACC_CSV = _find_csv("combined_main_ACC.csv")
RAW_CSV = _find_csv("combined_main_RAW.csv")   # plain CSV — no zip required
OUT_DIR = str(pathlib.Path(ACC_CSV).parent / "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"ACC_CSV : {ACC_CSV}")
print(f"RAW_CSV : {RAW_CSV}")
print(f"OUT_DIR : {OUT_DIR}")

# ------------------------------ DATA ------------------------------
SOURCE   = "RAW"            # "RAW"  -> accelerometer + gyroscope (recommended, more discriminative)
                            # "ACC"  -> accelerometer only (smaller / faster)
SENSORS  = ("acc", "gyro")  # used when SOURCE == "RAW". Add "magn" only if you really want it:
                            # magnetometer = compass heading, a strong *recording-specific* shortcut.
if SOURCE == "ACC":
    SENSORS = ("acc",)

FS          = 100           # resampling rate (Hz). Original ~200 Hz; gait energy is < 20 Hz.
LOWPASS_HZ  = 20
WIN_SEC     = 2.0           # window length (≈ 2 gait cycles)
STRIDE_SEC  = 0.5
MAX_GAP_SEC = 0.25          # break a recording into segments where timestamps jump
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15    # temporal blocks inside each recording; the rest is test
VOTE_K      = 10            # consecutive test windows combined in "block voting"  (≈ 7 s)

SEED = 42
DEBUG_MAX_CHUNKS = None     # e.g. 2 → read only 2 chunks of the RAW csv for a quick dry run

WIN, STRIDE = int(WIN_SEC * FS), int(STRIDE_SEC * FS)
GAP_WINS = int(math.ceil(WIN / STRIDE)) - 1      # windows to drop between blocks → zero overlap
np.random.seed(SEED); torch.manual_seed(SEED)
print(f"WIN={WIN} samples, STRIDE={STRIDE}, gap windows between splits={GAP_WINS}")

## 2. Load data (long format: one row = one sample of one sensor)

In [ ]:
def load_long():
    base = ["user_label", "record_id", "local_timestamp", "data_x", "data_y", "data_z"]
    if SOURCE == "ACC":
        cols = ["user_label", "record_id", "local_timestamp", "acc_x", "acc_y", "acc_z",
                "building_id", "condition", "condition_detail"]
        df = pd.read_csv(ACC_CSV, usecols=cols, low_memory=False)
        df = df.rename(columns={"acc_x": "data_x", "acc_y": "data_y", "acc_z": "data_z"})
        df["sensor_type"] = "acc"
    else:
        cols = base + ["sensor_type", "building_id", "condition", "condition_detail"]
        parts = []
        # RAW_CSV is a plain CSV on disk — no zip decompression needed
        reader = pd.read_csv(RAW_CSV, usecols=cols, chunksize=1_500_000,
                             encoding="utf-8-sig", low_memory=False)
        for i, ch in enumerate(reader):
            ch = ch[ch["sensor_type"].isin(SENSORS)].copy()
            for c in ["data_x", "data_y", "data_z"]:
                ch[c] = ch[c].astype("float32")
            ch["user_label"] = ch["user_label"].astype("int16")
            parts.append(ch)
            print(f"  chunk {i}: kept {len(ch):,} rows", end="\r")
            if DEBUG_MAX_CHUNKS and i + 1 >= DEBUG_MAX_CHUNKS:
                break
        df = pd.concat(parts, ignore_index=True); del parts
    # integer recording id (user + record_id), drop heavy strings
    df["rec"] = pd.factorize(df["user_label"].astype(str) + "|" + df["record_id"].astype(str))[0].astype("int32")
    meta = (df.groupby("rec").agg(user_label=("user_label", "first"), record_id=("record_id", "first"),
                                  building_id=("building_id", "first"), condition=("condition", "first"),
                                  condition_detail=("condition_detail", "first")).reset_index())
    df = df.drop(columns=["record_id", "building_id", "condition", "condition_detail"])
    df["sensor_type"] = df["sensor_type"].astype("category")
    return df, meta

t0 = time.time()
df, rec_meta = load_long()
print(f"\nrows={len(df):,}  recordings={df.rec.nunique()}  users={df.user_label.nunique()}  ({time.time()-t0:.0f}s)")
print(df.sensor_type.value_counts().to_dict())
rec_meta.head()

## 3. Clean → resample → filter → window

* sort by timestamp, drop duplicate timestamps (the raw files are slightly out of order)
* split a recording wherever the time jump is > `MAX_GAP_SEC`
* linear-interpolate every sensor onto a **uniform 100 Hz grid**
* zero-phase 20 Hz Butterworth low-pass (removes sensor noise, keeps gait)
* channels = `x, y, z, ‖v‖` per sensor (the magnitude is orientation-independent)

In [ ]:
SOS = butter(4, LOWPASS_HZ, btype="low", fs=FS, output="sos")
AXES = ["data_x", "data_y", "data_z"]
CH_NAMES = [f"{s}_{a}" for s in SENSORS for a in ("x", "y", "z", "m")]
N_CH = len(CH_NAMES)

def _sensor_arrays(g):
    out = {}
    for s in SENSORS:
        gs = g[g.sensor_type == s]
        if len(gs) < 100:
            return None
        gs = gs.sort_values("local_timestamp").drop_duplicates("local_timestamp")
        t = gs["local_timestamp"].values.astype(np.float64) / 1e9          # ns -> s
        out[s] = (t, gs[AXES].values.astype(np.float64))
    return out

def record_to_windows(g):
    # returns array (n_windows, N_CH, WIN) in temporal order, or None
    arr = _sensor_arrays(g)
    if arr is None:
        return None
    # common clock; if sensors do not overlap, align each to its own start
    t_start = max(v[0][0] for v in arr.values()); t_end = min(v[0][-1] for v in arr.values())
    if t_end - t_start < WIN_SEC:
        arr = {s: (t - t[0], x) for s, (t, x) in arr.items()}
        t_start = 0.0; t_end = min(v[0][-1] for v in arr.values())
    t_ref = arr[SENSORS[0]][0]
    t_ref = t_ref[(t_ref >= t_start) & (t_ref <= t_end)]
    if len(t_ref) < 2:
        return None
    cuts = np.where(np.diff(t_ref) > MAX_GAP_SEC)[0]
    bounds = np.concatenate([[0], cuts + 1, [len(t_ref)]])
    wins = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        ta, tb = t_ref[a], t_ref[b - 1]
        if tb - ta < WIN_SEC:
            continue
        grid = np.arange(ta, tb, 1.0 / FS)
        chans = []
        for s in SENSORS:
            t, x = arr[s]
            v = np.stack([np.interp(grid, t, x[:, k]) for k in range(3)], 0)   # (3, N)
            v = sosfiltfilt(SOS, v, axis=-1)
            chans += [v, np.linalg.norm(v, axis=0, keepdims=True)]
        sig = np.concatenate(chans, 0).astype(np.float32)                       # (N_CH, N)
        if sig.shape[1] < WIN:
            continue
        w = sliding_window_view(sig, WIN, axis=1)[:, ::STRIDE, :]               # (C, nW, WIN)
        wins.append(np.ascontiguousarray(w.transpose(1, 0, 2)))
    return np.concatenate(wins, 0) if wins else None

# sanity check: effective sampling rate per recording (timestamp unit check)
chk = df[df.sensor_type == SENSORS[0]].groupby("rec")["local_timestamp"].agg(["min", "max", "size"])
hz = (chk["size"] - 1) / ((chk["max"] - chk["min"]) / 1e9)
print("effective Hz per recording -> min/median/max:", hz.min().round(1), hz.median().round(1), hz.max().round(1))

In [ ]:
# ---- quick look: class balance + a sample window ----
cnt = pd.Series(y_all[split == 0]).value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
ax[0].bar(range(len(cnt)), cnt.values); ax[0].set_title("Train windows per user (class imbalance)"); ax[0].set_xlabel("class id")
i = np.where(split == 0)[0][0]
t = np.arange(WIN) / FS
for c in range(min(4, N_CH)):
    ax[1].plot(t, X_all[i, c], label=CH_NAMES[c])
ax[1].legend(); ax[1].set_title("Example 2-s window (accelerometer)"); ax[1].set_xlabel("s")
plt.tight_layout(); plt.show()

# class-imbalance handling (used by the models below)
class_counts = np.bincount(y_all[split == 0], minlength=N_CLASSES).astype(np.float64)
sample_w = (class_counts.sum() / (N_CLASSES * class_counts))[y_all]        # balanced weights per window
print("min/max train windows per user:", int(class_counts.min()), int(class_counts.max()))

## 4. Train / val / test tensors

Standardisation statistics are computed on the **training windows only** (no leakage).

In [ ]:
tr, va, te = [np.where(split == k)[0] for k in (0, 1, 2)]
ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
rec_te = rec_all[te]; widx_te = widx[te]

mu = X_all[tr].mean(axis=(0, 2), keepdims=True); sd = X_all[tr].std(axis=(0, 2), keepdims=True) + 1e-6
X_norm = ((X_all - mu) / sd).astype(np.float32)           # for deep learning
print("train/val/test:", len(tr), len(va), len(te))

def to_np(a):
    try:
        import cupy as cp
        if isinstance(a, cp.ndarray): return cp.asnumpy(a)
    except Exception:
        pass
    if hasattr(a, "to_numpy"): return a.to_numpy()
    return np.asarray(a)

## 5. GPU feature extraction (for classical ML)

Per channel: mean, std, min/max, quartiles, RMS, skewness, kurtosis, mean-abs-diff, zero-crossing rate, spectral band energies (0.5–20 Hz), dominant frequency, spectral entropy/centroid, and **gait autocorrelation** (peak height + lag = step regularity / cadence). Plus axis-to-axis correlations per sensor. Everything is computed with `torch` on the GPU.

In [ ]:
BANDS = [(0.5, 1), (1, 2), (2, 3), (3, 5), (5, 10), (10, 20)]

@torch.no_grad()
def _feat_batch(x):
    B, C, T = x.shape
    mean = x.mean(-1); xc = x - mean[..., None]
    std = xc.pow(2).mean(-1).sqrt() + 1e-6
    z = xc / std[..., None]
    q = torch.quantile(x, torch.tensor([.25, .5, .75], device=x.device), dim=-1)      # (3,B,C)
    feats = {
        "mean": mean, "std": std, "min": x.amin(-1), "max": x.amax(-1),
        "q25": q[0], "q50": q[1], "q75": q[2], "rms": x.pow(2).mean(-1).sqrt(),
        "skew": (z ** 3).mean(-1), "kurt": (z ** 4).mean(-1),
        "mad": (x[..., 1:] - x[..., :-1]).abs().mean(-1),
        "zcr": ((xc[..., 1:] * xc[..., :-1]) < 0).float().mean(-1),
    }
    P = torch.fft.rfft(xc, dim=-1).abs() ** 2
    fr = torch.fft.rfftfreq(T, 1.0 / FS).to(x.device)
    tot = P.sum(-1) + 1e-8; pn = P / tot[..., None]
    for a, b in BANDS:
        feats[f"band{a}-{b}"] = P[..., (fr >= a) & (fr < b)].sum(-1) / tot
    feats["spec_ent"] = -(pn * torch.log(pn + 1e-12)).sum(-1)
    feats["spec_cent"] = (pn * fr).sum(-1)
    feats["dom_freq"] = fr[P[..., 1:].argmax(-1) + 1]
    P2 = torch.fft.rfft(xc, n=2 * T, dim=-1).abs() ** 2                      # zero-padded -> linear autocorr
    ac = torch.fft.irfft(P2, n=2 * T, dim=-1)[..., :T]; ac = ac / (ac[..., :1] + 1e-8)
    lo, hi = int(0.3 * FS), int(1.5 * FS)
    acp, acl = ac[..., lo:hi].max(-1)
    feats["ac_peak"] = acp; feats["ac_lag"] = (acl + lo).float() / FS
    out, names = [], []
    for k, v in feats.items():
        out.append(v); names += [f"{k}|{c}" for c in CH_NAMES]
    for si, s in enumerate(SENSORS):                                          # cross-axis correlations
        o = si * 4
        for (i, j, nm) in [(0, 1, "xy"), (0, 2, "xz"), (1, 2, "yz")]:
            out.append((z[:, o + i] * z[:, o + j]).mean(-1, keepdim=True)); names.append(f"corr_{nm}|{s}")
    f = torch.cat([o_.reshape(B, -1) for o_ in out], 1)
    return torch.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0), names

@torch.no_grad()
def gpu_features(X, bs=1024):
    res = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(DEVICE).float()
        f, names = _feat_batch(xb); res.append(f.cpu())
    return torch.cat(res).numpy(), names

t0 = time.time()
F_all, FEAT_NAMES = gpu_features(X_all)
print("features:", F_all.shape, f"({time.time()-t0:.1f}s)")
scaler = StandardScaler().fit(F_all[tr])
Fs_all = scaler.transform(F_all).astype(np.float32)           # standardized (for LR / SVM / kNN / MLP)
Ftr, Fva, Fte = F_all[tr], F_all[va], F_all[te]               # raw (for tree models)
Str, Sva, Ste = Fs_all[tr], Fs_all[va], Fs_all[te]

## 6. Metrics helper + classical ML models (all on GPU)

In [ ]:
RESULTS, PROBA = [], {}          # PROBA[name] = {"va": ..., "te": ...}  (for ensembling)

def topk_acc(P, y, k=5):
    k = min(k, P.shape[1]); idx = np.argpartition(-P, k - 1, axis=1)[:, :k]
    return float((idx == y[:, None]).any(1).mean())

def block_vote(P, rec, widx, K=VOTE_K):
    # average class-probabilities over K consecutive windows of the same recording -> one decision per block
    order = np.lexsort((widx, rec)); P, rec_o = P[order], rec[order]
    ytrue = None; preds, rows = [], []
    start = 0
    out_p, out_r = [], []
    for r in np.unique(rec_o):
        idx = np.where(rec_o == r)[0]
        for s in range(0, len(idx), K):
            blk = idx[s:s + K]
            out_p.append(P[blk].mean(0)); out_r.append(r)
    return np.array(out_p), np.array(out_r)

def report(name, Pva, Pte, fit_time=np.nan):
    # Pva/Pte: class-probability matrices (N, N_CLASSES)
    pv, pt = Pva.argmax(1), Pte.argmax(1)
    row = dict(model=name, val_acc=accuracy_score(yva, pv), test_acc=accuracy_score(yte, pt),
               test_top5=topk_acc(Pte, yte), test_macroF1=f1_score(yte, pt, average="macro"), fit_s=fit_time)
    bp, br = block_vote(Pte, rec_te, widx_te)
    user_of_rec = {r: u for r, u in zip(rec_all, y_all)}
    row["test_blockvote_acc"] = float((bp.argmax(1) == np.array([user_of_rec[r] for r in br])).mean())
    RESULTS.append(row); PROBA[name] = dict(va=Pva, te=Pte)
    print(f"{name:28s} val={row['val_acc']:.4f}  TEST={row['test_acc']:.4f}  top5={row['test_top5']:.4f} "
          f"F1={row['test_macroF1']:.4f}  block-vote={row['test_blockvote_acc']:.4f}  ({fit_time:.0f}s)")

def onehot(pred, n=N_CLASSES):
    P = np.zeros((len(pred), n), np.float32); P[np.arange(len(pred)), pred.astype(int)] = 1; return P

In [ ]:
# ---------- XGBoost (GPU) ----------
import xgboost as xgb
t0 = time.time()
xgb_clf = xgb.XGBClassifier(
    n_estimators=1200, learning_rate=0.08, max_depth=6, subsample=0.8, colsample_bytree=0.6,
    min_child_weight=2, reg_lambda=2.0, objective="multi:softprob", eval_metric="mlogloss",
    tree_method="hist", device="cuda" if DEVICE.type == "cuda" else "cpu",
    early_stopping_rounds=40, random_state=SEED)
xgb_clf.fit(Ftr, ytr, sample_weight=sample_w[tr], eval_set=[(Fva, yva)], verbose=False)
report("XGBoost (GPU)", xgb_clf.predict_proba(Fva), xgb_clf.predict_proba(Fte), time.time() - t0)
print("best iteration:", xgb_clf.best_iteration)

In [ ]:
# ---------- cuML models (GPU). Skipped gracefully if cuML is not installed ----------
try:
    from cuml.ensemble import RandomForestClassifier as cuRF
    from cuml.linear_model import LogisticRegression as cuLR
    from cuml.svm import SVC as cuSVC
    from cuml.neighbors import KNeighborsClassifier as cuKNN
    HAS_CUML = True
except Exception as e:
    HAS_CUML = False
    print("cuML not available ->", type(e).__name__, "(install cuml-cu12 to enable GPU RF / SVM / kNN / LogReg)")

def run_cuml(name, make, Xt, Xv, Xe, proba=True):
    t0 = time.time()
    m = make(); m.fit(Xt.astype(np.float32), ytr.astype(np.int32))
    if proba:
        Pv, Pe = to_np(m.predict_proba(Xv.astype(np.float32))), to_np(m.predict_proba(Xe.astype(np.float32)))
    else:
        Pv, Pe = onehot(to_np(m.predict(Xv.astype(np.float32)))), onehot(to_np(m.predict(Xe.astype(np.float32))))
    report(name, Pv, Pe, time.time() - t0)

if HAS_CUML:
    run_cuml("cuML RandomForest", lambda: cuRF(n_estimators=400, max_depth=24, n_bins=128, random_state=SEED, n_streams=1), Ftr, Fva, Fte)
    run_cuml("cuML LogisticRegression", lambda: cuLR(max_iter=3000, C=1.0), Str, Sva, Ste)
    run_cuml("cuML SVC (RBF)", lambda: cuSVC(kernel="rbf", C=10.0, gamma="scale"), Str, Sva, Ste, proba=False)
    run_cuml("cuML kNN (k=5)", lambda: cuKNN(n_neighbors=5), Str, Sva, Ste)

In [ ]:
# ---------- MLP on hand-crafted features (pure PyTorch, GPU) ----------
def train_mlp(Xt, yt, Xv, yv, Xe, epochs=120, bs=512, lr=2e-3, wd=1e-3, patience=20):
    d = Xt.shape[1]
    net = nn.Sequential(nn.Linear(d, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.3),
                        nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.3),
                        nn.Linear(256, N_CLASSES)).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(Xt) / bs))
    Xt_, yt_ = torch.from_numpy(Xt).to(DEVICE), torch.from_numpy(yt).long().to(DEVICE)
    Xv_, Xe_ = torch.from_numpy(Xv).to(DEVICE), torch.from_numpy(Xe).to(DEVICE)
    cw = torch.tensor(class_counts.sum() / (N_CLASSES * class_counts), dtype=torch.float32, device=DEVICE)
    best, best_state, bad = -1, None, 0
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(Xt_), device=DEVICE)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            if len(b) < 2: continue
            xb = Xt_[b] + 0.05 * torch.randn_like(Xt_[b])
            loss = F.cross_entropy(net(xb), yt_[b], weight=cw, label_smoothing=0.05)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        net.eval()
        with torch.no_grad(): acc = (net(Xv_).argmax(1).cpu().numpy() == yv).mean()
        if acc > best: best, best_state, bad = acc, copy.deepcopy(net.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(best_state); net.eval()
    with torch.no_grad():
        return F.softmax(net(Xv_), 1).cpu().numpy(), F.softmax(net(Xe_), 1).cpu().numpy()

t0 = time.time()
Pv, Pe = train_mlp(Str, ytr, Sva, yva, Ste)
report("MLP on features (GPU)", Pv, Pe, time.time() - t0)

## 7. Deep learning on the raw windows (GPU, mixed precision)

Two architectures: **1D-ResNet** and **CNN + BiGRU**. Augmentation (jitter, per-channel scaling, random time shift, small time masking) is applied on the GPU; labels are smoothed; best epoch chosen on validation accuracy.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, 7, stride, 3, bias=False); self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, 7, 1, 3, bias=False);     self.b2 = nn.BatchNorm1d(cout)
        self.sk = nn.Sequential() if (cin == cout and stride == 1) else nn.Sequential(
            nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))
    def forward(self, x):
        o = F.gelu(self.b1(self.c1(x))); o = self.b2(self.c2(o)); return F.gelu(o + self.sk(x))

class ResNet1D(nn.Module):
    def __init__(self, cin, ncls, w=(64, 128, 256), drop=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(cin, w[0], 7, 1, 3, bias=False), nn.BatchNorm1d(w[0]), nn.GELU())
        layers, c = [], w[0]
        for i, wi in enumerate(w):
            layers += [ResBlock(c, wi, 1 if i == 0 else 2), ResBlock(wi, wi, 1)]; c = wi
        self.body = nn.Sequential(*layers); self.drop = nn.Dropout(drop); self.fc = nn.Linear(2 * c, ncls)
    def forward(self, x):
        h = self.body(self.stem(x)); h = torch.cat([h.mean(-1), h.amax(-1)], 1)
        return self.fc(self.drop(h))

class CNNBiGRU(nn.Module):
    def __init__(self, cin, ncls, hid=128, drop=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(cin, 64, 5, 1, 2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, 128, 5, 2, 2), nn.BatchNorm1d(128), nn.GELU(),
            nn.Conv1d(128, 128, 5, 2, 2), nn.BatchNorm1d(128), nn.GELU())
        self.gru = nn.GRU(128, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.drop = nn.Dropout(drop); self.fc = nn.Linear(4 * hid, ncls)
    def forward(self, x):
        h = self.cnn(x).transpose(1, 2); h, _ = self.gru(h)
        h = torch.cat([h.mean(1), h.amax(1)], 1); return self.fc(self.drop(h))

def augment(x):
    B, C, T = x.shape
    x = x * (1 + 0.08 * torch.randn(B, C, 1, device=x.device))              # per-channel scale
    x = x + 0.03 * torch.randn_like(x)                                      # jitter
    shift = int(torch.randint(-20, 21, (1,)))                               # random time shift
    x = torch.roll(x, shift, dims=-1)
    if torch.rand(1).item() < 0.5:                                          # short time mask
        L = int(torch.randint(5, 25, (1,))); s = int(torch.randint(0, T - L, (1,))); x = x.clone(); x[..., s:s + L] = 0
    return x

In [ ]:
Xtr_g = torch.from_numpy(X_norm[tr]).to(DEVICE); ytr_g = torch.from_numpy(ytr).long().to(DEVICE)
Xva_g = torch.from_numpy(X_norm[va]).to(DEVICE); Xte_g = torch.from_numpy(X_norm[te]).to(DEVICE)
cw_t = torch.tensor(class_counts.sum() / (N_CLASSES * class_counts), dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def predict_proba_dl(model, X, bs=1024):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            out.append(F.softmax(model(X[i:i + bs]).float(), 1).cpu())
    return torch.cat(out).numpy()

def train_dl(name, model, epochs=60, bs=256, lr=2e-3, wd=1e-2, patience=15):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * math.ceil(len(Xtr_g) / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.2)
    scaler_amp = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best, best_state, bad, t0 = -1, None, 0, time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr_g), device=DEVICE); tl = 0
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            if len(b) < 4: sched.step(); continue
            xb = augment(Xtr_g[b])
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                loss = F.cross_entropy(model(xb), ytr_g[b], weight=cw_t, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True); scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler_amp.step(opt); scaler_amp.update(); sched.step(); tl += loss.item() * len(b)
        Pv = predict_proba_dl(model, Xva_g); acc = (Pv.argmax(1) == yva).mean()
        if acc > best: best, best_state, bad = acc, copy.deepcopy(model.state_dict()), 0
        else: bad += 1
        if ep % 5 == 0 or bad == 0:
            print(f"[{name}] ep {ep:02d} loss {tl/len(Xtr_g):.4f} val_acc {acc:.4f} best {best:.4f}")
        if bad >= patience: print("early stop"); break
    model.load_state_dict(best_state)
    report(name, predict_proba_dl(model, Xva_g), predict_proba_dl(model, Xte_g), time.time() - t0)
    return model

resnet = train_dl("1D-ResNet (GPU)", ResNet1D(N_CH, N_CLASSES))

In [ ]:
bigru = train_dl("CNN-BiGRU (GPU)", CNNBiGRU(N_CH, N_CLASSES))

## 8. Model comparison, ensemble, per-user analysis

In [ ]:
# ---- soft-voting ensemble of the strongest models (chosen by VALIDATION accuracy) ----
res = pd.DataFrame(RESULTS).sort_values("val_acc", ascending=False)
top_names = [n for n in res.model if "kNN" not in n and "SVC" not in n][:4]      # models with real probabilities
Pva_e = np.mean([PROBA[n]["va"] for n in top_names], 0); Pte_e = np.mean([PROBA[n]["te"] for n in top_names], 0)
report("Ensemble: " + "+".join(n.split(" ")[0] for n in top_names), Pva_e, Pte_e)

res = pd.DataFrame(RESULTS).sort_values("test_acc", ascending=False).reset_index(drop=True)
display(res.style.format({c: "{:.4f}" for c in ["val_acc", "test_acc", "test_top5", "test_macroF1", "test_blockvote_acc"]} | {"fit_s": "{:.0f}"}))
res.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 0.5 * len(res) + 1.5))
ax.barh(res.model[::-1], res.test_acc[::-1], label="window accuracy")
ax.scatter(res.test_blockvote_acc[::-1], res.model[::-1], c="k", zorder=3, label=f"block vote (K={VOTE_K})")
ax.set_xlim(max(0, res.test_acc.min() - 0.05), 1.0); ax.set_xlabel("test accuracy"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
best_name = res.model.iloc[0]; Pbest = PROBA[best_name]["te"]; pbest = Pbest.argmax(1)
print("Best on test:", best_name, f"-> {accuracy_score(yte, pbest):.4f}")

# per-user accuracy
per_user = pd.DataFrame({"user": le.inverse_transform(yte), "ok": pbest == yte}).groupby("user").ok.mean().sort_values()
print("\nWorst 10 users:\n", per_user.head(10).round(3).to_string())

# accuracy by building / condition
mm = rec_meta.set_index("rec")
cond = pd.DataFrame({"cond": mm.loc[rec_te, "condition"].values, "bld": mm.loc[rec_te, "building_id"].values, "ok": pbest == yte})
print("\nBy condition:\n", cond.groupby("cond").ok.agg(["mean", "size"]).round(4).to_string())
print("\nBy building:\n", cond.groupby("bld").ok.agg(["mean", "size"]).round(4).to_string())

cm = confusion_matrix(yte, pbest, labels=np.arange(N_CLASSES), normalize="true")
plt.figure(figsize=(9, 8)); plt.imshow(cm, cmap="viridis", vmin=0, vmax=1); plt.colorbar(label="recall")
plt.title(f"Normalized confusion matrix – {best_name}"); plt.xlabel("predicted user idx"); plt.ylabel("true user idx"); plt.show()

## 9. Cross-recording sanity check (the honest generalisation test)

Only 5 users have more than one recording. For those users we **train on some recordings and test on a recording the model has never seen** (different session / condition). If this number is much lower than the in-recording accuracy above, the high score above is partly "recognising the recording" rather than purely "recognising the person".

In [ ]:
multi = pd.Series(rec_all).groupby(y_all).nunique(); multi_users = multi[multi >= 2].index.values
print("users with >=2 recordings:", len(multi_users), "->", le.inverse_transform(multi_users))
if len(multi_users) >= 2:
    m_idx = np.where(np.isin(y_all, multi_users))[0]
    held = {}
    for u in multi_users:
        recs_u = sorted(np.unique(rec_all[y_all == u])); held[u] = recs_u[-1]       # hold out last recording of each user
    is_test = np.array([rec_all[i] == held[y_all[i]] for i in m_idx])
    tr_i, te_i = m_idx[~is_test], m_idx[is_test]
    remap = {u: k for k, u in enumerate(multi_users)}
    ytr_m = np.array([remap[v] for v in y_all[tr_i]]); yte_m = np.array([remap[v] for v in y_all[te_i]])
    clf = xgb.XGBClassifier(n_estimators=400, learning_rate=0.08, max_depth=6, subsample=0.8, colsample_bytree=0.6,
                            tree_method="hist", device="cuda" if DEVICE.type == "cuda" else "cpu", random_state=SEED)
    clf.fit(F_all[tr_i], ytr_m)
    pm = clf.predict(F_all[te_i])
    print(f"cross-RECORDING accuracy on {len(multi_users)} multi-session users: {(pm == yte_m).mean():.4f}  "
          f"({len(te_i)} windows from unseen recordings)")
    print("chance level:", round(1 / len(multi_users), 3))
else:
    print("Not enough multi-recording users for this check.")

## 10. Save everything

In [ ]:
import joblib
torch.save(resnet.state_dict(), f"{OUT_DIR}/resnet1d.pt"); torch.save(bigru.state_dict(), f"{OUT_DIR}/cnn_bigru.pt")
xgb_clf.save_model(f"{OUT_DIR}/xgb_gpu.json")
joblib.dump(dict(scaler=scaler, le=le, mu=mu, sd=sd, cfg=dict(FS=FS, WIN=WIN, STRIDE=STRIDE, SENSORS=SENSORS,
            LOWPASS_HZ=LOWPASS_HZ, CH_NAMES=CH_NAMES, FEAT_NAMES=FEAT_NAMES)), f"{OUT_DIR}/preproc.joblib")
imp = pd.Series(xgb_clf.feature_importances_, index=FEAT_NAMES).sort_values(ascending=False)
print("Top-15 XGBoost features:\n", imp.head(15).round(4).to_string())
print("\nSaved to", OUT_DIR, os.listdir(OUT_DIR))

### Next steps to push accuracy further
* Add `"magn"` to `SENSORS` (watch for the shortcut caveat above), or add more windows with `STRIDE_SEC=0.25`.
* Longer windows (`WIN_SEC=3–4`) and more epochs for the DL models; try a Transformer/TCN encoder or a Siamese / metric-learning head (ArcFace) if you later need to identify *unseen* users.
* Train on several conditions and test on a held-out condition (e.g. train `relaxed_arm`, test `fixed_arm`) for a tougher benchmark.